
# AI PDF Chatbot — Ask Questions, Get Answers With Page References

**Project | AI Mini Projects Series**

**Problem:** Reading a long PDF to find one answer wastes a lot of time.

**What this notebook does:** You upload a PDF, ask questions in plain English, and the bot answers using only the PDF content and tells you which page the answer came from.

**How it works (RAG):** PDF → text per page → small chunks → embeddings → find the closest chunks to your question → send them to an LLM (Groq) → answer + page numbers.

**Tools:** Python, pypdf, sentence-transformers, numpy, Groq (openai/gpt-oss-120b), Google Colab

## Step 1: Install libraries

In [1]:
!pip install -q pypdf sentence-transformers openai reportlab

## Step 2: Import libraries and enter Groq key

In [2]:
import os
import getpass
import textwrap
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
model_name = "openai/gpt-oss-120b"
print("Groq client ready")

Enter your Groq API key: ··········
Groq client ready


## Step 3: Upload your PDF (or use the sample PDF)
If you cancel the upload, the notebook makes a small sample PDF (a company leave policy) so you can still test everything.

In [3]:
pdf_path = None
using_sample = False

try:
    from google.colab import files
    print("Upload your PDF now (cancel to use the sample PDF)")
    uploaded = files.upload()
    if len(uploaded) > 0:
        pdf_path = list(uploaded.keys())[0]
except Exception as e:
    print("Upload not available, using sample PDF")

if pdf_path is None:
    using_sample = True
    pdf_path = "sample_policy.pdf"
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import A4

    sample_pages = [
        "ABC Company Employee Handbook. Working hours are 9:30 AM to 6:30 PM, Monday to Friday. There is a one hour lunch break at 1:00 PM. Employees must mark attendance on the portal every day. Late arrival more than 3 times in a month will be counted as one half day leave.",
        "Leave Policy. Every employee gets 18 paid leaves per year. 12 days of sick leave are given every year and a medical certificate is needed if sick leave is more than 2 days. Maternity leave is 26 weeks and paternity leave is 10 days. Unused paid leaves can be carried forward up to 10 days.",
        "Work From Home Policy. Employees can work from home 2 days in a week after taking manager approval. Work from home is not allowed in the first 3 months of joining. Employees must be available on chat and video calls during working hours.",
        "Expense Reimbursement. Travel expenses for office work are reimbursed within 15 days. Bills must be submitted within 30 days of the trip. Hotel limit is 4000 rupees per night and food limit is 800 rupees per day. Expenses without bills will not be paid."
    ]

    c = canvas.Canvas(pdf_path, pagesize=A4)
    for page_text in sample_pages:
        lines = textwrap.wrap(page_text, 90)
        y = 800
        for line in lines:
            c.drawString(50, y, line)
            y = y - 18
        c.showPage()
    c.save()
    print("Sample PDF created:", pdf_path)

print("Using PDF:", pdf_path)

Upload your PDF now (cancel to use the sample PDF)


Saving NovaCart_Annual_Report_Test.pdf to NovaCart_Annual_Report_Test.pdf
Using PDF: NovaCart_Annual_Report_Test.pdf


## Step 4: Read the PDF page by page
We keep the text of each page separately so we can tell the page number later.

In [4]:
reader = PdfReader(pdf_path)
pages = []

for i in range(len(reader.pages)):
    text = reader.pages[i].extract_text()
    if text is None:
        text = ""
    pages.append(text)

print("Total pages:", len(pages))

total_chars = 0
for p in pages:
    total_chars = total_chars + len(p.strip())

if total_chars == 0:
    print("No text found. This PDF may be a scanned image, so this bot cannot read it.")
else:
    print("Total characters read:", total_chars)
    print("First 300 characters of page 1:")
    print(pages[0][:300])

Total pages: 5
Total characters read: 3060
First 300 characters of page 1:
NovaCart Annual Report 2025-26
1. Company Overview
NovaCart is an online shopping company started in 2019 in Bhopal, Madhya Pradesh. It sells
electronics, home products and fashion items across India. The company was founded by Riya
Sharma and Karan Mehta. The head office is in Bhopal and there are 


## Step 5: Split text into small chunks
Each chunk remembers which page it came from.

In [5]:
chunks = []
chunk_pages = []
chunk_size = 800
overlap = 100

for i in range(len(pages)):
    text = pages[i].strip()
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size]
        chunks.append(piece)
        chunk_pages.append(i + 1)
        if start + chunk_size >= len(text):
            break
        start = start + chunk_size - overlap

print("Total chunks:", len(chunks))

Total chunks: 5


## Step 6: Make embeddings
Embeddings turn text into numbers so we can find which chunks are closest to the question.

In [6]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embed_model.encode(chunks, normalize_embeddings=True)
print("Embeddings shape:", chunk_vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (5, 384)


## Step 7: Question answering function
Find the top 4 closest chunks, send them to Groq, and return the answer with the retrieved pages. Each page comes with a similarity score, best match first. The answer itself also mentions the page it used.

In [7]:
def ask_pdf(question, top_k=4):
    q_vector = embed_model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vector
    top_ids = np.argsort(scores)[::-1][:top_k]

    context = ""
    page_scores = {}
    for idx in top_ids:
        page_no = chunk_pages[idx]
        context = context + "[Page " + str(page_no) + "]\n" + chunks[idx] + "\n\n"
        # first time we see a page it has its best score, because top_ids is sorted
        if page_no not in page_scores:
            page_scores[page_no] = round(float(scores[idx]), 2)

    system_msg = "You answer questions only using the PDF text given. If the answer is not in the text, say 'I could not find this in the PDF'. Keep the answer short and clear. Mention the page number like (Page 2)."
    user_msg = "PDF text:\n" + context + "\nQuestion: " + question

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.2
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = "Error calling Groq: " + str(e)

    # best match first, with similarity score
    sources = []
    for page_no in page_scores:
        sources.append("Page " + str(page_no) + " (score " + str(page_scores[page_no]) + ")")
    return answer, sources

## Step 8: Test with some questions

In [8]:
if using_sample:
    demo_questions = [
        "How many paid leaves does an employee get in a year?",
        "How many days can I work from home in a week?",
        "What is the hotel limit for travel?"
    ]
else:
    demo_questions = [
        "What is this document about?",
        "Give me the 3 most important points from this document."
    ]

for q in demo_questions:
    answer, sources = ask_pdf(q)
    print("Q:", q)
    print("A:", answer)
    print("Retrieved (best first):", sources)
    print("-" * 60)

Q: What is this document about?
A: The document is NovaCart’s Annual Report 2025‑26, which outlines the company’s overview, sales performance, team and hiring details, and plans for 2026‑27 (Page 1).
Retrieved (best first): ['Page 5 (score 0.07)', 'Page 1 (score 0.04)', 'Page 4 (score 0.01)', 'Page 2 (score -0.01)']
------------------------------------------------------------
Q: Give me the 3 most important points from this document.
A: - NovaCart earned 480 crore rupees in revenue for 2025‑26, a 32 % increase over the previous year (Page 2).  
- The company aims to reach 600 crore rupees in revenue in 2026‑27 and will open a third warehouse in Nagpur by September 2026 (Page 5).  
- A new AI shopping assistant and Hindi/Marathi language options will be launched in the app by December 2026, with a technology spend of 35 crore rupees (Page 5).
Retrieved (best first): ['Page 5 (score 0.09)', 'Page 2 (score 0.06)', 'Page 4 (score 0.05)', 'Page 1 (score 0.05)']
-----------------------------

## Step 8b: Test with questions that are NOT in the PDF
A good bot should not guess. These questions have nothing to do with the PDF, so the answer should be 'I could not find this in the PDF'.

In [9]:
outside_questions = [
    "What is the capital of Australia?",
    "Who won the FIFA World Cup in 2018?"
]

for q in outside_questions:
    answer, sources = ask_pdf(q)
    print("Q:", q)
    print("A:", answer)
    print("-" * 60)

Q: What is the capital of Australia?
A: I could not find this in the PDF.
------------------------------------------------------------
Q: Who won the FIFA World Cup in 2018?
A: I could not find this in the PDF.
------------------------------------------------------------


## Step 9: Chat with your PDF
Type your own questions. Type `exit` to stop.

In [10]:
while True:
    q = input("Ask a question (type exit to stop): ")
    if q.strip().lower() == "exit":
        print("Bye!")
        break
    if q.strip() == "":
        continue
    answer, sources = ask_pdf(q)
    print("Answer:", answer)
    print("Retrieved (best first):", sources)
    print("-" * 60)

Ask a question (type exit to stop): Who founded NovaCart?
Answer: NovaCart was founded by Riya Sharma and Karan Mehta. (Page 1)
Retrieved (best first): ['Page 1 (score 0.59)', 'Page 5 (score 0.4)', 'Page 4 (score 0.34)', 'Page 3 (score 0.28)']
------------------------------------------------------------
Ask a question (type exit to stop): What was the total revenue in 2025-26?
Answer: The total revenue in 2025‑26 was **480 crore rupees** (Page 2).
Retrieved (best first): ['Page 2 (score 0.61)', 'Page 5 (score 0.35)', 'Page 4 (score 0.3)', 'Page 1 (score 0.23)']
------------------------------------------------------------
Ask a question (type exit to stop): Which category sold the most?
Answer: Electronics was the best‑selling category (Page 2).
Retrieved (best first): ['Page 2 (score 0.42)', 'Page 3 (score 0.3)', 'Page 1 (score 0.22)', 'Page 5 (score 0.15)']
------------------------------------------------------------
Ask a question (type exit to stop): What was the best month?
Answer:

## Key Points

- The bot only answers from the PDF text, so answers are grounded in the document
- Every answer mentions the page it came from, and the retrieved pages are shown with similarity scores (best match first)
- If the answer is not in the PDF, the bot is told to say "I could not find this in the PDF". Step 8b tests this with questions that have nothing to do with the document
- Scanned PDFs (images) do not work because there is no text to read. OCR would be needed for those

**Ideas to improve:** add OCR for scanned PDFs, keep chat history for follow-up questions, build a Streamlit app and deploy it.

---
Built by Akshat Kesharwani | GitHub: https://github.com/akshatkesharwani-info | Portfolio: https://akshatkesharwani-info.github.io/